# Caso 3 · El cierre de octubre (abierto)
**30 minutos · en parejas · sin pistas por defecto**

> **De:** Dirección de Finanzas
> **Asunto:** Cierre de octubre, otra vez
>
> Después de lo de octubre acordamos **una sola definición**: venta bruta **sin IVA**, por **fecha local** del pedido.
> Aun así: Comercial (almacén de datos) **$4,141,430.88**; Finanzas (pasarela + cortes de caja, al corte del 3 de noviembre a las 9:00) **$4,033,258.92**.
> Necesito saber qué arreglamos, qué documentamos y qué decidimos. Una página.

### Lo que te dieron
| Tabla | Origen | Lo que dice el diccionario |
|---|---|---|
| `pedidos` | DW | `total`: total del pedido. `cargado_en`: cuándo llegó al DW |
| `lineas_pedido` | DW | `precio_unitario` sin IVA |
| `productos` | DW | catálogo vigente |
| `clientes` | DW | un renglón por cliente. `cargado_en`: fecha de carga |
| `pasarela_cobros` | Pasarela de pagos | `monto`: monto del cobro. `estado`: aprobado / rechazado |
| `caja_sucursal` | Punto de venta | `importe`: importe del ticket. `recibido_en`: cuándo llegó el corte |
| `cargas` | Bitácora | cargas de cortes de caja |

**Entregable:** el puente de conciliación (residuo 0) y el informe de una página al final del cuaderno.
Revisa tus montos con `nnc.revisar_hallazgo("caso3", monto)`: te dice si coincide con una causa, sin decir cuál.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (funciona en Colab y en local)
import importlib.util, os, sys, subprocess, pathlib
REPO_URL = "https://github.com/jona866/taller-conciliacion-de-datos-datostada2026.git"
if "google.colab" in sys.modules:
    destino = pathlib.Path("/content/taller-conciliacion-de-datos-datostada2026")
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(destino)], check=True)
    os.chdir(destino)
    if any(importlib.util.find_spec(mod) is None for mod in ("duckdb", "pandas", "pytz", "pytest")):
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    if not pathlib.Path("datos").exists():
        subprocess.run([sys.executable, "generar_datos.py"], check=True, stdout=subprocess.DEVNULL)
else:
    raiz = pathlib.Path.cwd()
    while not (raiz / "nnc.py").exists() and raiz != raiz.parent:
        raiz = raiz.parent
    if not (raiz / "nnc.py").exists():
        raise FileNotFoundError("Abre el cuaderno desde el repositorio del taller.")
    os.chdir(raiz)
if not pathlib.Path("datos/manifiesto.json").exists():
    raise FileNotFoundError("Faltan los datos. Ejecuta python generar_datos.py en la raíz del repositorio.")
sys.path.insert(0, os.getcwd())
import nnc
con = nnc.conectar("caso3")
print("Listo ✅  Tablas:", ", ".join(t for (t,) in con.execute("SHOW TABLES").fetchall()))

In [ ]:
comercial = nnc.total(con, "caso3_dashboard_ventas")
finanzas = nnc.total(con, "caso3_reporte_finanzas")
diferencia = comercial - finanzas
print(f"Comercial {comercial:,.2f} · Finanzas {finanzas:,.2f} · Diferencia {diferencia:,.2f}")

In [ ]:
nnc.ver_sql("caso3_dashboard_ventas")
print("=" * 70)
nnc.ver_sql("caso3_reporte_finanzas")

### Bitácora de hipótesis
| # | Hipótesis | Consulta que la prueba o la descarta | Resultado | Efecto ($) |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

<details><summary>💡 Pista 1 · Método</summary>

Compara en el **mismo grano**: arma una tabla por `pedido_id` con lo que dice el DW y lo que dicen las fuentes operativas, y únelas con `FULL OUTER JOIN`.

</details>

<details><summary>💡 Pista 2 · Antes de comparar</summary>

Revisa **unidades** y **qué significa cada columna**. ¿`monto` está en pesos? ¿`total` trae IVA? Compruébalo con datos, no con el diccionario.

</details>

<details><summary>💡 Pista 3 · El dashboard</summary>

Cuenta filas después de **cada** JOIN del dashboard. Si cambia el conteo, busca la llave repetida o la llave que no encuentra pareja.

</details>

<details><summary>💡 Pista 4 · El corte</summary>

Finanzas cerró con lo que había recibido a las 9:00 del 3 de noviembre. ¿El DW tiene lo mismo?

</details>

<details><summary>💡 Pista 5 · Centavos</summary>

Una diferencia real puede ser inocua. Cuantifícala igual y documéntala.

</details>

### Tu investigación

_(agrega las celdas que necesites)_

In [ ]:
# Cuando tengas todas tus causas, arma el puente (efecto sobre comercial − finanzas, con signo)
nnc.puente({
    "Causa 1": 0,
    "Causa 2": 0,
}, diferencia)

## Informe de una página

**Diferencia:** comercial − finanzas = $

| Causa | Evidencia (consulta) | Efecto | ¿Arreglar, documentar o decidir con negocio? |
|---|---|---|---|
| | | | |

**Cifra recomendada para el cierre y por qué:**

**Pruebas que agregamos:**